# 00 — How the locked model was trained (optional, for transparency)

This is the exact code that produced the locked model in `models/TrustBreast_locked/`
(80/20 split with `random_state=42`, 91-patient validation carve-out, SMOTE on the 364 model-fit
patients only, ×3 Gaussian-noise augmentation, RF + XGBoost + DNN, thresholds tuned on validation).

**You do not need to run it.** Notebooks 01–04 load the saved model, so every reported number is
reproduced exactly. Re-training the DNN on different hardware or library versions is not bit-exact,
so a re-trained model may differ slightly from the locked one. The locked model's held-out accuracy
is 99.12%; across 30 repeated splits the same pipeline gives 97.19 ± 1.65% (notebook 01), which is
the representative figure reported in the paper.

The SAVE cell writes to `models/TrustBreast_locked_retrained/` and never overwrites the locked model.

In [ ]:
# ============================================
# CELL 0 — DETERMINISM  (run this FIRST, before any imports)
# Adds three settings that make the DNN identical across runs:
#   1. os.environ flags   -> make GPU/cuDNN deterministic (must be set BEFORE imports)
#   2. all seeds          -> python / numpy / tensorflow
#   3. enable_op_determinism() -> LOCKS GPU floating-point order (this was the missing piece)
# The reseed() helper is used later to reset the RNG right before the DNN is built.
# ============================================
import os
os.environ['PYTHONHASHSEED']         = '42'
os.environ['TF_DETERMINISTIC_OPS']   = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

import random, numpy as np, tensorflow as tf
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    print("enable_op_determinism() ON  ->  DNN now reproducible")
except Exception as e:
    print("Note: enable_op_determinism unavailable (older TF). The remaining fixes still apply.")

def reseed(s=SEED):
    random.seed(s); np.random.seed(s); tf.random.set_seed(s)

print("TF:", tf.__version__, "| Determinism setup done. Now run the remaining cells.")


In [ ]:
!pip -q install scikit-learn xgboost imbalanced-learn tensorflow scipy shap lime dice-ml anthropic matplotlib seaborn

In [ ]:
# ============================================
# CORRECTED — LEAKAGE-FREE (Point 1 fix)
# Data to ensemble — a single cell
# TWO FIXES:
#   (1) Thresholds are now chosen on VALIDATION (not on test)
#   (2) The DNN now early-stops on VALIDATION (not on test)
# All output variable names are unchanged — the O2/O3/O4 cells run without modification
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import warnings
warnings.filterwarnings('ignore')
tf.random.set_seed(42)
np.random.seed(42)

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, classification_report)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.initializers import GlorotUniform

# ===== THRESHOLD SELECTION OBJECTIVE =====
# 'accuracy' = same as the original method (only the leakage removed, otherwise unchanged).
# 'f1'       = clinically preferable (focuses on the malignant minority). Change here to switch.
OBJECTIVE = 'f1'

def pick_threshold(proba_val, y_val_, objective='accuracy'):
    """Choose the threshold on VALIDATION data (the test set is never seen)."""
    best_t, best_s = 0.5, -1.0
    for t in np.arange(0.25, 0.75, 0.01):
        pred = (proba_val >= t).astype(int)
        s = f1_score(y_val_, pred) if objective == 'f1' else accuracy_score(y_val_, pred)
        if s > best_s:
            best_s, best_t = s, t
    return best_t

# ─── 1. DATA ────────────────────────────────
url = ("https://archive.ics.uci.edu/ml/"
       "machine-learning-databases/"
       "breast-cancer-wisconsin/wdbc.data")
col_names = ['id','diagnosis',
  'radius_mean','texture_mean','perimeter_mean','area_mean',
  'smoothness_mean','compactness_mean','concavity_mean','concave_points_mean',
  'symmetry_mean','fractal_dimension_mean',
  'radius_se','texture_se','perimeter_se','area_se','smoothness_se','compactness_se',
  'concavity_se','concave_points_se','symmetry_se','fractal_dimension_se',
  'radius_worst','texture_worst','perimeter_worst','area_worst',
  'smoothness_worst','compactness_worst','concavity_worst','concave_points_worst',
  'symmetry_worst','fractal_dimension_worst']

df = pd.read_csv(url, header=None, names=col_names)
le = LabelEncoder()
df['diagnosis'] = le.fit_transform(df['diagnosis'])   # B=0, M=1
X = df.drop(['id','diagnosis'], axis=1)
y = df['diagnosis']

print("="*50); print("DATA LOADED"); print("="*50)
print(f"Total: {len(df)} patients | B: {sum(y==0)} | M: {sum(y==1)}")

# ─── 2. SPLIT (569 → 455 train + 114 test) ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)
print(f"\nTrain: {len(y_train)} | Test: {len(y_test)}  (test set is not used again until final evaluation)")

# ─── 3. NORMALIZE (fit on train, transform test) ──
scaler = MinMaxScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# ─── 3b. VALIDATION CARVE-OUT (BEFORE SMOTE) ──
# 455 train → 364 fit + 91 validation (real patients, stratified)
X_fit_sc, X_val_sc, y_fit, y_val = train_test_split(
    X_train_sc, y_train, test_size=0.20, random_state=42, stratify=y_train)
print(f"\nValidation split: fit={len(y_fit)} | val={len(y_val)} (real, untouched)")
print(f"  val → B:{sum(y_val==0)} M:{sum(y_val==1)}")

# ─── 4. SMOTE (FIT set only) ─────────────
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_fit_sc, y_fit)
print(f"\nSMOTE (fit only): B={sum(y_train_sm==0)} M={sum(y_train_sm==1)}")

# ─── 5. DATA AUGMENTATION (fit/SMOTE data only) ──
def add_noise(X, noise=0.01):
    n = np.random.normal(0, noise, X.shape)
    return np.clip(X + n, 0, 1)

X_aug1 = add_noise(X_train_sm, 0.01)
X_aug2 = add_noise(X_train_sm, 0.02)
X_train_aug = np.vstack([X_train_sm, X_aug1, X_aug2])
y_train_aug = np.hstack([y_train_sm, y_train_sm, y_train_sm])
idx = np.random.permutation(len(X_train_aug))
X_train_aug = X_train_aug[idx]; y_train_aug = y_train_aug[idx]
print(f"Augmented: {len(X_train_aug)} samples")

# ─── 6. RANDOM FOREST ───────────────────────
print("\n" + "="*50); print("RF TRAINING..."); print("="*50)
rf_model = RandomForestClassifier(
    n_estimators=500, max_depth=None, random_state=42, n_jobs=1)   # n_jobs=1 for determinism
rf_model.fit(X_train_aug, y_train_aug)

rf_prob_raw = rf_model.predict_proba(X_test_sc)[:, 1]      # TEST probs (for evaluation)
rf_prob_val = rf_model.predict_proba(X_val_sc)[:, 1]       # VAL probs (for threshold selection)
best_rf_thr = pick_threshold(rf_prob_val, y_val, OBJECTIVE)  # ← chosen on VAL
best_rf_pred = (rf_prob_raw >= best_rf_thr).astype(int)      # ← FROZEN thr, applied to test once

acc_rf  = accuracy_score(y_test, best_rf_pred)
prec_rf = precision_score(y_test, best_rf_pred)
rec_rf  = recall_score(y_test, best_rf_pred)
f1_rf   = f1_score(y_test, best_rf_pred)
auc_rf  = roc_auc_score(y_test, rf_prob_raw)
cm_rf   = confusion_matrix(y_test, best_rf_pred)
tn_rf,fp_rf,fn_rf,tp_rf = cm_rf.ravel(); spec_rf = tn_rf/(tn_rf+fp_rf)
print(f"RF thr={best_rf_thr:.2f} | Acc {acc_rf*100:.2f}% | Recall {rec_rf:.4f} | F1 {f1_rf:.4f} | AUC {auc_rf:.4f} | Miss {fn_rf}")

# ─── 7. XGBOOST ─────────────────────────────
print("\n" + "="*50); print("XGBOOST TRAINING..."); print("="*50)
xgb_model = XGBClassifier(
    learning_rate=0.01, max_depth=4, n_estimators=500,
    subsample=0.9, colsample_bytree=0.9, random_state=42,
    eval_metric='logloss', verbosity=0)
xgb_model.fit(X_train_aug, y_train_aug)

xgb_prob_raw = xgb_model.predict_proba(X_test_sc)[:, 1]
xgb_prob_val = xgb_model.predict_proba(X_val_sc)[:, 1]
best_xgb_thr = pick_threshold(xgb_prob_val, y_val, OBJECTIVE)
best_xgb_pred = (xgb_prob_raw >= best_xgb_thr).astype(int)

acc_xgb  = accuracy_score(y_test, best_xgb_pred)
prec_xgb = precision_score(y_test, best_xgb_pred)
rec_xgb  = recall_score(y_test, best_xgb_pred)
f1_xgb   = f1_score(y_test, best_xgb_pred)
auc_xgb  = roc_auc_score(y_test, xgb_prob_raw)
cm_xgb   = confusion_matrix(y_test, best_xgb_pred)
tn_xgb,fp_xgb,fn_xgb,tp_xgb = cm_xgb.ravel(); spec_xgb = tn_xgb/(tn_xgb+fp_xgb)
print(f"XGB thr={best_xgb_thr:.2f} | Acc {acc_xgb*100:.2f}% | Recall {rec_xgb:.4f} | F1 {f1_xgb:.4f} | AUC {auc_xgb:.4f} | Miss {fn_xgb}")

# ─── 8. DNN ─────────────────────────────────
print("\n" + "="*50); print("DNN TRAINING..."); print("="*50)
reseed()   # RF/XGB consumed the RNG -> restore a fixed state before building the DNN
dnn_model = Sequential([
    Dense(1024, activation='relu', kernel_regularizer=l2(0.0005),
          kernel_initializer=GlorotUniform(seed=42), input_shape=(30,)),
    BatchNormalization(), Dropout(0.4),
    Dense(512, activation='relu', kernel_regularizer=l2(0.0005),
          kernel_initializer=GlorotUniform(seed=43)),
    BatchNormalization(), Dropout(0.4),
    Dense(256, activation='relu', kernel_regularizer=l2(0.0005),
          kernel_initializer=GlorotUniform(seed=44)),
    BatchNormalization(), Dropout(0.3),
    Dense(128, activation='relu', kernel_regularizer=l2(0.0005),
          kernel_initializer=GlorotUniform(seed=45)),
    BatchNormalization(), Dropout(0.3),
    Dense(64, activation='relu', kernel_initializer=GlorotUniform(seed=46)),
    BatchNormalization(), Dropout(0.2),
    Dense(32, activation='relu', kernel_initializer=GlorotUniform(seed=47)),
    BatchNormalization(), Dropout(0.2),
    Dense(1, activation='sigmoid', kernel_initializer=GlorotUniform(seed=48))
])
dnn_model.compile(optimizer=Adam(learning_rate=0.001),
                  loss='binary_crossentropy', metrics=['accuracy'])
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=25,
                  restore_best_weights=True, mode='max'),
    ModelCheckpoint('best_dnn.keras', monitor='val_accuracy',
                    save_best_only=True, mode='max')
]
history = dnn_model.fit(
    X_train_aug, y_train_aug,
    epochs=150, batch_size=16,
    validation_data=(X_val_sc, y_val),
    shuffle=False,   # ← FIX: VALIDATION (not test)
    callbacks=callbacks, verbose=0)
print(f"DNN best val acc: {max(history.history['val_accuracy'])*100:.2f}%  (this is VAL, not test)")

dnn_best = tf.keras.models.load_model('best_dnn.keras')
dnn_prob_raw = dnn_best.predict(X_test_sc, verbose=0).flatten()
dnn_prob_val = dnn_best.predict(X_val_sc, verbose=0).flatten()
best_dnn_thr = pick_threshold(dnn_prob_val, y_val, OBJECTIVE)
best_dnn_pred = (dnn_prob_raw >= best_dnn_thr).astype(int)

acc_d  = accuracy_score(y_test, best_dnn_pred)
prec_d = precision_score(y_test, best_dnn_pred)
rec_d  = recall_score(y_test, best_dnn_pred)
f1_d   = f1_score(y_test, best_dnn_pred)
auc_d  = roc_auc_score(y_test, dnn_prob_raw)
cm_d   = confusion_matrix(y_test, best_dnn_pred)
tn_d,fp_d,fn_d,tp_d = cm_d.ravel(); spec_d = tn_d/(tn_d+fp_d)
print(f"DNN thr={best_dnn_thr:.2f} | Acc {acc_d*100:.2f}% | Recall {rec_d:.4f} | F1 {f1_d:.4f} | AUC {auc_d:.4f} | Miss {fn_d}")

# ─── 9. SOFT VOTING ENSEMBLE ────────────────
print("\n" + "="*50); print("SOFT VOTING ENSEMBLE"); print("="*50)
prob_ensemble     = (rf_prob_raw + xgb_prob_raw + dnn_prob_raw) / 3   # TEST
prob_ensemble_val = (rf_prob_val + xgb_prob_val + dnn_prob_val) / 3   # VAL
best_ens_thr = pick_threshold(prob_ensemble_val, y_val, OBJECTIVE)    # ← chosen on VAL
ens_pred = (prob_ensemble >= best_ens_thr).astype(int)               # ← FROZEN, applied to test once
print(f"Ensemble threshold (chosen on validation): {best_ens_thr:.2f}")

acc_e  = accuracy_score(y_test, ens_pred)
prec_e = precision_score(y_test, ens_pred)
rec_e  = recall_score(y_test, ens_pred)
f1_e   = f1_score(y_test, ens_pred)
auc_e  = roc_auc_score(y_test, prob_ensemble)
cm_e   = confusion_matrix(y_test, ens_pred)
tn_e,fp_e,fn_e,tp_e = cm_e.ravel(); spec_e = tn_e/(tn_e+fp_e)

# ─── 10. FINAL RESULTS TABLE ────────────────
print("\n" + "="*65); print("COMPLETE RESULTS — ALL 4 MODELS (LEAKAGE-FREE)"); print("="*65)
print(f"{'Metric':<14} {'RF':>10} {'XGBoost':>10} {'DNN':>10} {'Ensemble':>10}")
print("-"*56)
all_m = [
    ('Accuracy%',  acc_rf*100, acc_xgb*100, acc_d*100, acc_e*100),
    ('Precision',  prec_rf, prec_xgb, prec_d, prec_e),
    ('Recall',     rec_rf, rec_xgb, rec_d, rec_e),
    ('F1-Score',   f1_rf, f1_xgb, f1_d, f1_e),
    ('AUC-ROC',    auc_rf, auc_xgb, auc_d, auc_e),
    ('Specificity',spec_rf, spec_xgb, spec_d, spec_e),
    ('Miss(FN)',   fn_rf, fn_xgb, fn_d, fn_e),
]
for name, rv, xv, dv, ev in all_m:
    if name == 'Accuracy%':
        print(f"{name:<14} {rv:>9.2f}% {xv:>9.2f}% {dv:>9.2f}% {ev:>9.2f}%")
    elif name == 'Miss(FN)':
        print(f"{name:<14} {int(rv):>10} {int(xv):>10} {int(dv):>10} {int(ev):>10}")
    else:
        print(f"{name:<14} {rv:>10.4f} {xv:>10.4f} {dv:>10.4f} {ev:>10.4f}")

print(f"\nChosen thresholds → RF:{best_rf_thr:.2f} XGB:{best_xgb_thr:.2f} DNN:{best_dnn_thr:.2f} ENS:{best_ens_thr:.2f}")
print("Note: thresholds were chosen on VALIDATION; the test set was used only for final evaluation.")

# ─── 11. VISUALIZATION (same as before) ─────
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
ConfusionMatrixDisplay(cm_e, display_labels=['Benign','Malignant']).plot(
    ax=axes[0][0], cmap='Greens', colorbar=False)
axes[0][0].set_title(f'Ensemble Confusion Matrix\n{acc_e*100:.2f}%', fontsize=12, fontweight='bold')
for prob, nm, col in [(rf_prob_raw,f'RF ({auc_rf:.3f})','blue'),
                      (xgb_prob_raw,f'XGB ({auc_xgb:.3f})','orange'),
                      (dnn_prob_raw,f'DNN ({auc_d:.3f})','purple'),
                      (prob_ensemble,f'Ensemble ({auc_e:.3f})','green')]:
    RocCurveDisplay.from_predictions(y_test, prob, name=nm, ax=axes[0][1], color=col)
axes[0][1].plot([0,1],[0,1],'k--')
axes[0][1].set_title('ROC Curve — All Models', fontsize=12, fontweight='bold')
axes[0][1].legend(fontsize=8)
models = ['RF','XGBoost','DNN','Ensemble']
accs = [acc_rf*100, acc_xgb*100, acc_d*100, acc_e*100]
colors = ['#3498DB','#E67E22','#9B59B6','#27AE60']
bars = axes[1][0].bar(models, accs, color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, accs):
    axes[1][0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                    f'{val:.2f}%', ha='center', fontsize=10, fontweight='bold')
axes[1][0].set_ylim(90, 102); axes[1][0].set_ylabel('Accuracy %')
axes[1][0].set_title('4 Models Accuracy', fontsize=12, fontweight='bold')
axes[1][0].grid(axis='y', alpha=0.3)
f1s = [f1_rf, f1_xgb, f1_d, f1_e]
bars2 = axes[1][1].bar(models, f1s, color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars2, f1s):
    axes[1][1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                    f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')
axes[1][1].set_ylim(0.88, 1.02); axes[1][1].set_ylabel('F1 Score')
axes[1][1].set_title('4 Models F1 Score', fontsize=12, fontweight='bold')
axes[1][1].grid(axis='y', alpha=0.3)
plt.suptitle(f'LEAKAGE-FREE ENSEMBLE — {acc_e*100:.2f}%', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.savefig('final_ensemble_leakagefree.png', dpi=150); plt.show()

# ─── 12. PAPER PARAGRAPH ────────────────────
print("\n" + "="*60); print("PAPER PARAGRAPH (new numbers)"); print("="*60)
print(f"""
A soft-voting ensemble of Random Forest, XGBoost, and a Deep Neural Network,
with all decision thresholds selected on a held-out validation split (never the
test set), achieved on the 114-patient test set:

Accuracy:  {acc_e*100:.2f}%
Precision: {prec_e:.4f}
Recall:    {rec_e:.4f}
F1-Score:  {f1_e:.4f}
AUC-ROC:   {auc_e:.4f}
Miss:      {fn_e} cancer patient(s)
""")


In [ ]:
# ===== SAVE (run ONCE, right after O1 shows 99.12%) =====
import os, pickle, joblib
SAVE_DIR = 'models/TrustBreast_locked_retrained'   # locked model is never overwritten
os.makedirs(SAVE_DIR, exist_ok=True)

joblib.dump(rf_model,  SAVE_DIR + '/rf_model.pkl')
joblib.dump(xgb_model, SAVE_DIR + '/xgb_model.pkl')
joblib.dump(scaler,    SAVE_DIR + '/scaler.pkl')
dnn_best.save(SAVE_DIR + '/dnn_best.keras')
dnn_model.save(SAVE_DIR + '/dnn_model.keras')

state = dict(X=X, X_test=X_test, X_test_sc=X_test_sc, X_train=X_train,
             X_train_sc=globals().get('X_train_sc'), X_val_sc=X_val_sc,
             y_test=y_test, y_val=y_val, y_train=y_train,
             X_train_aug=X_train_aug, y_train_aug=y_train_aug,
             rf_prob_raw=rf_prob_raw, xgb_prob_raw=xgb_prob_raw, dnn_prob_raw=dnn_prob_raw,
             prob_ensemble=prob_ensemble, ens_pred=ens_pred,
             best_rf_thr=best_rf_thr, best_xgb_thr=best_xgb_thr,
             best_dnn_thr=best_dnn_thr, best_ens_thr=best_ens_thr)
with open(SAVE_DIR + '/state.pkl','wb') as f: pickle.dump(state, f)

from sklearn.metrics import accuracy_score
print('SAVED locked state to', SAVE_DIR)
print('Locked ensemble accuracy:', round(accuracy_score(y_test, ens_pred)*100, 2), 'percent')